In [1]:
from transformers import BertModel, BertTokenizer, AutoModelForSequenceClassification
import torch

#### Contextual Embeddings

In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
text1 = "I deposited money in the bank"
text2 = "I sat by the river bank"

inputs1 = tokenizer(text1, return_tensors="pt")
inputs2 = tokenizer(text2, return_tensors="pt")

In [5]:
outputs1 = model(**inputs1)
outputs2 = model(**inputs2)

In [9]:
bank_token_id = tokenizer.convert_tokens_to_ids("bank")

bank_idx1 = (inputs1['input_ids'][0] == bank_token_id).nonzero(as_tuple=True)[0][0]
bank_idx2 = (inputs2['input_ids'][0] == bank_token_id).nonzero(as_tuple=True)[0][0]

In [10]:
bank_emb1 = outputs1.last_hidden_state[0, bank_idx1, :]
bank_emb2 = outputs2.last_hidden_state[0, bank_idx2, :]

are_identical = torch.allclose(bank_emb1, bank_emb2)
print(f"Are the embeddings exactly the same? {are_identical}")

Are the embeddings exactly the same? False


In [11]:
similarity = torch.nn.functional.cosine_similarity(
    bank_emb1.unsqueeze(0),
    bank_emb2.unsqueeze(0),
)

print(f"Cosine Similarity: {similarity.item():.4f}")

Cosine Similarity: 0.7317


#### Model Quantization

In [12]:
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [13]:
quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

/tmp/ipykernel_2754/1225607595.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [14]:
print(f"Original model size: {model.get_memory_footprint() / 1e6:.2f} MB")

Original model size: 267.82 MB
